# <a id='toc1_'></a>[Imports](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Imports](#toc1_)    
- [Chargement des données](#toc2_)    
  - [Lecture des fichiers CSV](#toc2_1_)    
  - [Sélection du dataset](#toc2_2_)    
  - [Aperçu](#toc2_3_)    
- [Utilitaires](#toc3_)    
  - [Fonctions de logging MLflow](#toc3_1_)    
  - [Gestion des runs](#toc3_2_)    
  - [Pipeline sklearn](#toc3_3_)    
- [Préparation des données](#toc4_)    
  - [Features & target](#toc4_1_)    
  - [Statistiques du split](#toc4_2_)    
- [Configuration MLflow](#toc5_)    
- [Modélisation](#toc6_)    
  - [Fonction run_cv](#toc6_1_)    
  - [Entraînement des baselines](#toc6_2_)    
  - [Entraînement avec sample weights (balanced)](#toc6_3_)    
  - [Entraînement avec seuil optimisé (Fbeta β=1.5, CV-aware)](#toc6_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [44]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import lightgbm as lgb
from lightgbm import LGBMClassifier
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    confusion_matrix,
    make_scorer,
)
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_sample_weight
import mlflow
from mlflow import MlflowClient

# <a id='toc2_'></a>[Chargement des données](#toc0_)

## <a id='toc2_1_'></a>[Lecture des fichiers CSV](#toc0_)

In [45]:
df_full = pd.read_csv("df.csv")
df_red = pd.read_csv("df_red.csv")

## <a id='toc2_2_'></a>[Sélection du dataset](#toc0_)

In [46]:
use_reduced = True  # flip to False for full run
df = df_red.copy() if use_reduced else df_full.copy()

## <a id='toc2_3_'></a>[Aperçu](#toc0_)

In [47]:
df

,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,CC_SK_DPD_DEF_MEAN,CC_SK_DPD_DEF_SUM,CC_SK_DPD_DEF_VAR,CC_NAME_CONTRACT_STATUS_Active_MEAN,CC_NAME_CONTRACT_STATUS_Completed_MEAN,CC_NAME_CONTRACT_STATUS_Demand_MEAN,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_nan_MEAN,CC_COUNT
0,100002,1,0,0,0,0,202500.0,406597.5,24700.5,351000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0,1,0,1,0,270000.0,1293502.5,35698.5,1129500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0,0,1,0,0,67500.0,135000.0,6750.0,135000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0,1,0,0,0,135000.0,312682.5,29686.5,297000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,0,0,0,0,121500.0,513000.0,21865.5,513000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,134821,0,0,1,1,2,202500.0,738000.0,26275.5,738000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29996,134822,0,1,0,0,1,225000.0,1288350.0,37800.0,1125000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29997,134825,0,1,1,0,1,135000.0,269982.0,27792.0,238500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29998,134826,0,1,0,0,0,99000.0,247275.0,17338.5,225000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# <a id='toc3_'></a>[Utilitaires](#toc0_)

## <a id='toc3_1_'></a>[Fonctions de logging MLflow](#toc0_)

In [48]:
log_model_fn = {
    LGBMClassifier: mlflow.lightgbm.log_model,
    XGBClassifier: mlflow.xgboost.log_model,
}

## <a id='toc3_2_'></a>[Gestion des runs](#toc0_)

In [49]:
def delete_run_if_exists(experiment_name, run_name):
    client = MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        return
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
    )
    for run in runs:
        client.delete_run(run.info.run_id)

## <a id='toc3_3_'></a>[Pipeline sklearn](#toc0_)

In [50]:
def make_sklearn_pipeline(model):
    """Wrapping imputation + scaling pour les modèles sklearn qui ne gèrent pas les NaN."""
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", model),
        ]
    )

In [51]:
def cost_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return 10 * fn + fp  # positif = à minimiser


scorer = make_scorer(cost_score, greater_is_better=False)

In [52]:
def find_best_threshold(y_true, y_proba, n_steps=100):
    """Retourne le seuil qui minimise le coût métier (10*FN + FP)."""
    thresholds = np.linspace(0.01, 0.99, n_steps)
    costs = [cost_score(y_true, (y_proba >= t).astype(int)) for t in thresholds]
    best_idx = np.argmin(costs)
    return thresholds[best_idx]

# <a id='toc4_'></a>[Préparation des données](#toc0_)

## <a id='toc4_1_'></a>[Features & target](#toc0_)

In [53]:
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]


# Fix noms de colonnes pour LightGBM
import re

X.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", col) for col in X.columns]

# on enleve les inf pour eviter de faire bugger certains modeles
X = X.replace([np.inf, -np.inf], np.nan)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=78
)

## <a id='toc4_2_'></a>[Statistiques du split](#toc0_)

In [54]:
y_train.value_counts()

TARGET
0    22110
1     1890
Name: count, dtype: int64

In [55]:
y_test.value_counts()

TARGET
0    5487
1     513
Name: count, dtype: int64

# <a id='toc5_'></a>[Configuration MLflow](#toc0_)

In [56]:
if mlflow.active_run():
    mlflow.end_run()

mlflow.set_tracking_uri("http://127.0.0.1:5000/")
EXPERIMENT_NAME = "First trial"
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1774003960636, experiment_id='2', last_update_time=1774003960636, lifecycle_stage='active', name='First trial', tags={}, workspace='default'>

# <a id='toc6_'></a>[Modélisation](#toc0_)

## <a id='toc6_1_'></a>[Fonction run_cv](#toc0_)

In [ ]:
FBETA = 1.5


def run_cv(
    model,
    model_name,
    X,
    y,
    n_splits=5,
    tags=None,
    use_sample_weights=False,
    optimize_threshold=False,
):
    delete_run_if_exists(EXPERIMENT_NAME, model_name)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs, pr_aucs, recalls, precisions, f1s, fbetas, costs, thresholds = (
        [],
        [],
        [],
        [],
        [],
        [],
        [],
        [],
    )
    sw_key = "model__sample_weight" if isinstance(model, Pipeline) else "sample_weight"

    with mlflow.start_run(run_name=model_name, tags=tags or {}):
        mlflow.log_params({k: str(v) for k, v in model.get_params().items()})
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("sample_weight", "balanced" if use_sample_weights else "none")
        mlflow.log_param("optimize_threshold", str(optimize_threshold))
        base = model.steps[-1][1] if isinstance(model, Pipeline) else model
        mlflow.set_tag("model_family", type(base).__name__)
        for fold, (tr, val) in enumerate(skf.split(X, y)):
            sw = (
                {sw_key: compute_sample_weight("balanced", y.iloc[tr])}
                if use_sample_weights
                else {}
            )
            model.fit(X.iloc[tr], y.iloc[tr], **sw)
            preds_proba = model.predict_proba(X.iloc[val])[:, 1]
            if optimize_threshold:
                thresh = find_best_threshold(y.iloc[val], preds_proba)
                thresholds.append(thresh)
            else:
                thresh = 0.5
            preds = (preds_proba >= thresh).astype(int)
            aucs.append(roc_auc_score(y.iloc[val], preds_proba))
            pr_aucs.append(average_precision_score(y.iloc[val], preds_proba))
            recalls.append(recall_score(y.iloc[val], preds, zero_division=0))
            precisions.append(precision_score(y.iloc[val], preds, zero_division=0))
            f1s.append(f1_score(y.iloc[val], preds, zero_division=0))
            fbetas.append(fbeta_score(y.iloc[val], preds, beta=FBETA, zero_division=0))
            costs.append(cost_score(y.iloc[val], preds))
        mlflow.log_metric("roc_auc_mean", round(float(np.mean(aucs)), 2))
        mlflow.log_metric("roc_auc_std", round(float(np.std(aucs)), 2))
        mlflow.log_metric("pr_auc_mean", round(float(np.mean(pr_aucs)), 2))
        mlflow.log_metric("pr_auc_std", round(float(np.std(pr_aucs)), 2))
        mlflow.log_metric("recall_mean", round(float(np.mean(recalls)), 2))
        mlflow.log_metric("recall_std", round(float(np.std(recalls)), 2))
        mlflow.log_metric("precision_mean", round(float(np.mean(precisions)), 2))
        mlflow.log_metric("precision_std", round(float(np.std(precisions)), 2))
        mlflow.log_metric("f1_mean", round(float(np.mean(f1s)), 2))
        mlflow.log_metric("f1_std", round(float(np.std(f1s)), 2))
        mlflow.log_metric("fbeta_mean", round(float(np.mean(fbetas)), 2))
        mlflow.log_metric("fbeta_std", round(float(np.std(fbetas)), 2))
        mlflow.log_metric("cost_mean", round(float(np.mean(costs)), 2))
        mlflow.log_metric("cost_std", round(float(np.std(costs)), 2))
        if optimize_threshold:
            mlflow.log_metric(
                "best_threshold_mean", round(float(np.mean(thresholds)), 2)
            )
            mlflow.log_metric("best_threshold_std", round(float(np.std(thresholds)), 2))

        sw = (
            {sw_key: compute_sample_weight("balanced", y)} if use_sample_weights else {}
        )
        model.fit(X, y, **sw)
        log_fn = log_model_fn.get(type(model), mlflow.sklearn.log_model)
        log_fn(model, name=model_name)

        thr_info = (
            f" | Threshold={np.mean(thresholds):.3f} ± {np.std(thresholds):.3f}"
            if optimize_threshold
            else ""
        )
        print(
            f"{model_name} | ROC-AUC={np.mean(aucs):.4f} | PR-AUC={np.mean(pr_aucs):.4f} | "
            f"Recall={np.mean(recalls):.4f} | Precision={np.mean(precisions):.4f} | "
            f"Fbeta(β={FBETA})={np.mean(fbetas):.4f} | Cost={np.mean(costs):.1f}{thr_info}"
        )

## <a id='toc6_2_'></a>[Entraînement des baselines](#toc0_)

In [58]:
baseline_tags = {"dataset": "reduced", "phase": "baseline", "fbeta": str(FBETA)}

run_cv(
    LGBMClassifier(random_state=42, n_jobs=-1),
    "lgbm_baseline",
    X_train,
    y_train,
    tags=baseline_tags,
)
run_cv(
    XGBClassifier(random_state=42, n_jobs=-1, eval_metric="auc"),
    "xgb_baseline",
    X_train,
    y_train,
    tags=baseline_tags,
)
run_cv(
    make_sklearn_pipeline(
        MLPClassifier(
            random_state=42,
            hidden_layer_sizes=(100, 50),
            early_stopping=True,
            max_iter=300,
        )
    ),
    "MLP_baseline",
    X_train,
    y_train,
    tags=baseline_tags,
)
run_cv(
    make_sklearn_pipeline(
        LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
    ),
    "log_reg_baseline",
    X_train,
    y_train,
    tags=baseline_tags,
)
run_cv(
    make_sklearn_pipeline(RandomForestClassifier(random_state=42, n_jobs=-1)),
    "rf_baseline",
    X_train,
    y_train,
    tags=baseline_tags,
)

[LightGBM] [Info] Number of positive: 1512, number of negative: 17688
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,010225 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 40433
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 707
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0,078750 -> initscore=-2,459453
[LightGBM] [Info] Start training from score -2,459453
[LightGBM] [Info] Number of positive: 1512, number of negative: 17688
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,025692 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 40599
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 708
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0,078750 -> initscore=-2,459453
[LightGBM] [Info] Start training from score -2,459453
[LightGBM]

2026/03/30 17:24:59 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/30 17:25:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


lgbm_baseline | ROC-AUC=0.7257 | PR-AUC=0.2075 | Recall=0.0312 | Precision=0.4195 | Fbeta(β=1.5)=0.0436 | Cost=3678.4
🏃 View run lgbm_baseline at: http://127.0.0.1:5000/#/experiments/2/runs/3aadb34a7cd645afafd3f4dced2e1b54
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/03/30 17:25:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


xgb_baseline | ROC-AUC=0.6969 | PR-AUC=0.1808 | Recall=0.0487 | Precision=0.3234 | Fbeta(β=1.5)=0.0658 | Cost=3635.2
🏃 View run xgb_baseline at: http://127.0.0.1:5000/#/experiments/2/runs/7ea363ed94de4e0487e207e65b3f51db
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


...

MLP_baseline | ROC-AUC=0.6887 | PR-AUC=0.1705 | Recall=0.0079 | Precision=0.1457 | Fbeta(β=1.5)=0.0112 | Cost=3759.8
🏃 View run MLP_baseline at: http://127.0.0.1:5000/#/experiments/2/runs/10df0e5c363344219eb4f07f1a334b05
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE_SIZE_MEAN' 'BURO_STATUS_0_MEAN_MEAN'
 'BURO_STATUS_1_MEAN_MEAN' 'BURO_STATUS_2_MEAN_MEAN'
 'BURO_STATUS_3_MEAN_MEAN' 'BURO_STATUS_4_MEAN_MEAN'
 'BURO_STATUS_5_MEAN_MEAN' 'BURO_STATUS_C_MEAN_MEAN'
 'BURO_STATUS_X_MEAN_MEAN' 'BURO_STATUS_nan_MEAN_MEAN'
 'ACTIVE_MONTHS_BALANCE_MIN_MIN' 'ACTIVE_MONTHS_BALANCE_MAX_MAX'
 'ACTIVE_MONTHS_BALANCE_SIZE_MEAN' 'CLOSED_MONTHS_BALANCE_MIN_MIN'
 'CLOSED_MONTHS_BALANCE_MAX_MAX' 'CLOSED_MONTHS_BALANCE_SIZE_MEAN']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.

log_reg_baseline | ROC-AUC=0.7129 | PR-AUC=0.1843 | Recall=0.0286 | Precision=0.2673 | Fbeta(β=1.5)=0.0394 | Cost=3701.8
🏃 View run log_reg_baseline at: http://127.0.0.1:5000/#/experiments/2/runs/e82489d3c1bd45229e51de575d8d3a1f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE_SIZE_MEAN' 'BURO_STATUS_0_MEAN_MEAN'
 'BURO_STATUS_1_MEAN_MEAN' 'BURO_STATUS_2_MEAN_MEAN'
 'BURO_STATUS_3_MEAN_MEAN' 'BURO_STATUS_4_MEAN_MEAN'
 'BURO_STATUS_5_MEAN_MEAN' 'BURO_STATUS_C_MEAN_MEAN'
 'BURO_STATUS_X_MEAN_MEAN' 'BURO_STATUS_nan_MEAN_MEAN'
 'ACTIVE_MONTHS_BALANCE_MIN_MIN' 'ACTIVE_MONTHS_BALANCE_MAX_MAX'
 'ACTIVE_MONTHS_BALANCE_SIZE_MEAN' 'CLOSED_MONTHS_BALANCE_MIN_MIN'
 'CLOSED_MONTHS_BALANCE_MAX_MAX' 'CLOSED_MONTHS_BALANCE_SIZE_MEAN']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE

rf_baseline | ROC-AUC=0.6846 | PR-AUC=0.1670 | Recall=0.0005 | Precision=0.2000 | Fbeta(β=1.5)=0.0008 | Cost=3778.0
🏃 View run rf_baseline at: http://127.0.0.1:5000/#/experiments/2/runs/c7a2ede490af4f309bc6e2ed283fedbb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [59]:
# ### Entrainement sur le full

# baseline_tags = {"dataset": "full", "phase": "baseline", "fbeta": str(FBETA)}

# run_cv(
#     LGBMClassifier(random_state=42, n_jobs=-1),
#     "lgbm_baseline_full",
#     X_train,
#     y_train,
#     tags=baseline_tags,
# )
# run_cv(
#     XGBClassifier(random_state=42, n_jobs=-1, eval_metric="auc"),
#     "xgb_baseline_full",
#     X_train,
#     y_train,
#     tags=baseline_tags,
# )
# run_cv(
#     make_sklearn_pipeline(
#         MLPClassifier(
#             random_state=42,
#             hidden_layer_sizes=(100, 50),
#             early_stopping=True,
#             max_iter=300,
#         )
#     ),
#     "MLP_baseline_full",
#     X_train,
#     y_train,
#     tags=baseline_tags,
# )
# run_cv(
#     make_sklearn_pipeline(
#         LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
#     ),
#     "log_reg_baseline_full",
#     X_train,
#     y_train,
#     tags=baseline_tags,
# )
# run_cv(
#     make_sklearn_pipeline(RandomForestClassifier(random_state=42, n_jobs=-1)),
#     "rf_baseline_full",
#     X_train,
#     y_train,
#     tags=baseline_tags,
# )

## <a id='toc6_3_'></a>[Entraînement avec sample weights (balanced)](#toc0_)

MLP exclu : `MLPClassifier` ne supporte pas `sample_weight` dans `fit()`.

In [60]:
sw_tags = {"dataset": "reduced", "phase": "sample_weights", "fbeta": str(FBETA)}

run_cv(
    LGBMClassifier(random_state=42, n_jobs=-1),
    "lgbm_sw",
    X_train,
    y_train,
    tags=sw_tags,
    use_sample_weights=True,
)
run_cv(
    XGBClassifier(random_state=42, n_jobs=-1, eval_metric="auc"),
    "xgb_sw",
    X_train,
    y_train,
    tags=sw_tags,
    use_sample_weights=True,
)
run_cv(
    make_sklearn_pipeline(LogisticRegression(max_iter=1000, random_state=42)),
    "log_reg_sw",
    X_train,
    y_train,
    tags=sw_tags,
    use_sample_weights=True,
)
run_cv(
    make_sklearn_pipeline(RandomForestClassifier(random_state=42, n_jobs=-1)),
    "rf_sw",
    X_train,
    y_train,
    tags=sw_tags,
    use_sample_weights=True,
)

[LightGBM] [Info] Number of positive: 1512, number of negative: 17688
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,010398 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 40433
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 707
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0,500000 -> initscore=0,000000
[LightGBM] [Info] Start training from score 0,000000
[LightGBM] [Info] Number of positive: 1512, number of negative: 17688
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,014013 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 40599
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 708
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0,500000 -> initscore=0,000000
[LightGBM] [Info] Start training from score 0,000000
[LightGBM] [In

2026/03/30 17:26:16 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/30 17:26:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


lgbm_sw | ROC-AUC=0.7185 | PR-AUC=0.2052 | Recall=0.4593 | Precision=0.1930 | Fbeta(β=1.5)=0.3223 | Cost=2768.8
🏃 View run lgbm_sw at: http://127.0.0.1:5000/#/experiments/2/runs/4ee00fb101af45c8b46467af84eeb84a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/03/30 17:26:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


xgb_sw | ROC-AUC=0.6898 | PR-AUC=0.1815 | Recall=0.3026 | Precision=0.2029 | Fbeta(β=1.5)=0.2628 | Cost=3087.2
🏃 View run xgb_sw at: http://127.0.0.1:5000/#/experiments/2/runs/9f935ba5d4b547ff9dba5d25bb0b1608
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE_SIZE_MEAN' 'BURO_STATUS_0_MEAN_MEAN'
 'BURO_STATUS_1_MEAN_MEAN' 'BURO_STATUS_2_MEAN_MEAN'
 'BURO_STATUS_3_MEAN_MEAN' 'BURO_STATUS_4_MEAN_MEAN'
 'BURO_STATUS_5_MEAN_MEAN' 'BURO_STATUS_C_MEAN_MEAN'
 'BURO_STATUS_X_MEAN_MEAN' 'BURO_STATUS_nan_MEAN_MEAN'
 'ACTIVE_MONTHS_BALANCE_MIN_MIN' 'ACTIVE_MONTHS_BALANCE_MAX_MAX'
 'ACTIVE_MONTHS_BALANCE_SIZE_MEAN' 'CLOSED_MONTHS_BALANCE_MIN_MIN'
 'CLOSED_MONTHS_BALANCE_MAX_MAX' 'CLOSED_MONTHS_BALANCE_SIZE_MEAN']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE

log_reg_sw | ROC-AUC=0.7074 | PR-AUC=0.1779 | Recall=0.6090 | Precision=0.1485 | Fbeta(β=1.5)=0.3116 | Cost=2798.4
🏃 View run log_reg_sw at: http://127.0.0.1:5000/#/experiments/2/runs/81f62a5e0526442e9c139ac6d1ec5fc6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE_SIZE_MEAN' 'BURO_STATUS_0_MEAN_MEAN'
 'BURO_STATUS_1_MEAN_MEAN' 'BURO_STATUS_2_MEAN_MEAN'
 'BURO_STATUS_3_MEAN_MEAN' 'BURO_STATUS_4_MEAN_MEAN'
 'BURO_STATUS_5_MEAN_MEAN' 'BURO_STATUS_C_MEAN_MEAN'
 'BURO_STATUS_X_MEAN_MEAN' 'BURO_STATUS_nan_MEAN_MEAN'
 'ACTIVE_MONTHS_BALANCE_MIN_MIN' 'ACTIVE_MONTHS_BALANCE_MAX_MAX'
 'ACTIVE_MONTHS_BALANCE_SIZE_MEAN' 'CLOSED_MONTHS_BALANCE_MIN_MIN'
 'CLOSED_MONTHS_BALANCE_MAX_MAX' 'CLOSED_MONTHS_BALANCE_SIZE_MEAN']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE

rf_sw | ROC-AUC=0.6973 | PR-AUC=0.1750 | Recall=0.0011 | Precision=0.2667 | Fbeta(β=1.5)=0.0015 | Cost=3776.4
🏃 View run rf_sw at: http://127.0.0.1:5000/#/experiments/2/runs/b9cd0ebeca494a3ca534d854e1311c72
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [61]:
# ### Entrainement sur le full

# sw_tags = {"dataset": "full", "phase": "sample_weights", "fbeta": str(FBETA)}

# run_cv(
#     LGBMClassifier(random_state=42, n_jobs=-1),
#     "lgbm_sw_full",
#     X_train,
#     y_train,
#     tags=sw_tags,
#     use_sample_weights=True,
# )
# run_cv(
#     XGBClassifier(random_state=42, n_jobs=-1, eval_metric="auc"),
#     "xgb_sw_full",
#     X_train,
#     y_train,
#     tags=sw_tags,
#     use_sample_weights=True,
# )
# run_cv(
#     make_sklearn_pipeline(LogisticRegression(max_iter=1000, random_state=42)),
#     "log_reg_sw_full",
#     X_train,
#     y_train,
#     tags=sw_tags,
#     use_sample_weights=True,
# )
# run_cv(
#     make_sklearn_pipeline(RandomForestClassifier(random_state=42, n_jobs=-1)),
#     "rf_sw_full",
#     X_train,
#     y_train,
#     tags=sw_tags,
#     use_sample_weights=True,
# )

## <a id='toc6_4_'></a>[Entraînement avec seuil optimisé (Fbeta β=1.5, CV-aware)](#toc0_)

In [62]:
thr_tags = {"dataset": "reduced", "phase": "threshold_opt", "fbeta": str(FBETA)}

run_cv(
    LGBMClassifier(random_state=42, n_jobs=-1),
    "lgbm_thr",
    X_train,
    y_train,
    tags=thr_tags,
    use_sample_weights=True,
    optimize_threshold=True,
)
run_cv(
    XGBClassifier(random_state=42, n_jobs=-1, eval_metric="auc"),
    "xgb_thr",
    X_train,
    y_train,
    tags=thr_tags,
    use_sample_weights=True,
    optimize_threshold=True,
)
run_cv(
    make_sklearn_pipeline(LogisticRegression(max_iter=1000, random_state=42)),
    "log_reg_thr",
    X_train,
    y_train,
    tags=thr_tags,
    use_sample_weights=True,
    optimize_threshold=True,
)
run_cv(
    make_sklearn_pipeline(RandomForestClassifier(random_state=42, n_jobs=-1)),
    "rf_thr",
    X_train,
    y_train,
    tags=thr_tags,
    use_sample_weights=True,
    optimize_threshold=True,
)
# MLP : threshold optimisé sans sample_weight (non supporté par MLPClassifier)
run_cv(
    make_sklearn_pipeline(
        MLPClassifier(
            random_state=42,
            hidden_layer_sizes=(100, 50),
            early_stopping=True,
            max_iter=300,
        )
    ),
    "mlp_thr",
    X_train,
    y_train,
    tags=thr_tags,
    use_sample_weights=False,
    optimize_threshold=True,
)

[LightGBM] [Info] Number of positive: 1512, number of negative: 17688
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,009877 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 40433
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 707
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0,500000 -> initscore=0,000000
[LightGBM] [Info] Start training from score 0,000000
[LightGBM] [Info] Number of positive: 1512, number of negative: 17688
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,011199 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 40599
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 708
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0,500000 -> initscore=0,000000
[LightGBM] [Info] Start training from score 0,000000
[LightGBM] [In

2026/03/30 17:27:43 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/30 17:27:45 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


lgbm_thr | ROC-AUC=0.7185 | PR-AUC=0.2052 | Recall=0.6127 | Precision=0.1622 | Fbeta(β=1.5)=0.3268 | Cost=2701.0 | Threshold=0.382 ± 0.063
🏃 View run lgbm_thr at: http://127.0.0.1:5000/#/experiments/2/runs/33b7e9a042aa455ab35432d25c7ce5ba
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/03/30 17:27:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


xgb_thr | ROC-AUC=0.6898 | PR-AUC=0.1815 | Recall=0.5450 | Precision=0.1512 | Fbeta(β=1.5)=0.3020 | Cost=2877.8 | Threshold=0.273 ± 0.023
🏃 View run xgb_thr at: http://127.0.0.1:5000/#/experiments/2/runs/661efd6632824fb3b45ca8864c913813
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


...

log_reg_thr | ROC-AUC=0.7074 | PR-AUC=0.1779 | Recall=0.6011 | Precision=0.1587 | Fbeta(β=1.5)=0.3207 | Cost=2726.4 | Threshold=0.523 ± 0.049
🏃 View run log_reg_thr at: http://127.0.0.1:5000/#/experiments/2/runs/ecc7dfef53124cec8731cf55e1bdd455
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE_SIZE_MEAN' 'BURO_STATUS_0_MEAN_MEAN'
 'BURO_STATUS_1_MEAN_MEAN' 'BURO_STATUS_2_MEAN_MEAN'
 'BURO_STATUS_3_MEAN_MEAN' 'BURO_STATUS_4_MEAN_MEAN'
 'BURO_STATUS_5_MEAN_MEAN' 'BURO_STATUS_C_MEAN_MEAN'
 'BURO_STATUS_X_MEAN_MEAN' 'BURO_STATUS_nan_MEAN_MEAN'
 'ACTIVE_MONTHS_BALANCE_MIN_MIN' 'ACTIVE_MONTHS_BALANCE_MAX_MAX'
 'ACTIVE_MONTHS_BALANCE_SIZE_MEAN' 'CLOSED_MONTHS_BALANCE_MIN_MIN'
 'CLOSED_MONTHS_BALANCE_MAX_MAX' 'CLOSED_MONTHS_BALANCE_SIZE_MEAN']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE

rf_thr | ROC-AUC=0.6973 | PR-AUC=0.1750 | Recall=0.5698 | Precision=0.1492 | Fbeta(β=1.5)=0.3041 | Cost=2864.8 | Threshold=0.099 ± 0.006
🏃 View run rf_thr at: http://127.0.0.1:5000/#/experiments/2/runs/17c1783c5dc1419da4badded56629fc3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE_SIZE_MEAN' 'BURO_STATUS_0_MEAN_MEAN'
 'BURO_STATUS_1_MEAN_MEAN' 'BURO_STATUS_2_MEAN_MEAN'
 'BURO_STATUS_3_MEAN_MEAN' 'BURO_STATUS_4_MEAN_MEAN'
 'BURO_STATUS_5_MEAN_MEAN' 'BURO_STATUS_C_MEAN_MEAN'
 'BURO_STATUS_X_MEAN_MEAN' 'BURO_STATUS_nan_MEAN_MEAN'
 'ACTIVE_MONTHS_BALANCE_MIN_MIN' 'ACTIVE_MONTHS_BALANCE_MAX_MAX'
 'ACTIVE_MONTHS_BALANCE_SIZE_MEAN' 'CLOSED_MONTHS_BALANCE_MIN_MIN'
 'CLOSED_MONTHS_BALANCE_MAX_MAX' 'CLOSED_MONTHS_BALANCE_SIZE_MEAN']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Kevin\projects\OC_P6\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['BURO_MONTHS_BALANCE_MIN_MIN' 'BURO_MONTHS_BALANCE_MAX_MAX'
 'BURO_MONTHS_BALANCE

mlp_thr | ROC-AUC=0.6887 | PR-AUC=0.1705 | Recall=0.5513 | Precision=0.1562 | Fbeta(β=1.5)=0.3083 | Cost=2837.0 | Threshold=0.097 ± 0.012
🏃 View run mlp_thr at: http://127.0.0.1:5000/#/experiments/2/runs/890f0c744f6f48959e0d3677f66689cb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [63]:
# ### Entrainement sur le full

# thr_tags = {"dataset": "full", "phase": "threshold_opt", "fbeta": str(FBETA)}

# run_cv(
#     LGBMClassifier(random_state=42, n_jobs=-1),
#     "lgbm_thr_full",
#     X_train,
#     y_train,
#     tags=thr_tags,
#     use_sample_weights=True,
#     optimize_threshold=True,
# )
# run_cv(
#     XGBClassifier(random_state=42, n_jobs=-1, eval_metric="auc"),
#     "xgb_thr_full",
#     X_train,
#     y_train,
#     tags=thr_tags,
#     use_sample_weights=True,
#     optimize_threshold=True,
# )
# run_cv(
#     make_sklearn_pipeline(LogisticRegression(max_iter=1000, random_state=42)),
#     "log_reg_thr_full",
#     X_train,
#     y_train,
#     tags=thr_tags,
#     use_sample_weights=True,
#     optimize_threshold=True,
# )
# run_cv(
#     make_sklearn_pipeline(RandomForestClassifier(random_state=42, n_jobs=-1)),
#     "rf_thr_full",
#     X_train,
#     y_train,
#     tags=thr_tags,
#     use_sample_weights=True,
#     optimize_threshold=True,
# )
# # MLP : threshold optimisé sans sample_weight (non supporté par MLPClassifier)
# run_cv(
#     make_sklearn_pipeline(
#         MLPClassifier(
#             random_state=42,
#             hidden_layer_sizes=(100, 50),
#             early_stopping=True,
#             max_iter=300,
#         )
#     ),
#     "mlp_thr_full",
#     X_train,
#     y_train,
#     tags=thr_tags,
#     use_sample_weights=False,
#     optimize_threshold=True,
# )